# «Семейный щит»: установка сервера из Google Colab

Блокнот подключается к серверу `188.120.246.123` по SSH с серверов Google и запускает установщик
`shield/deploy/install.sh`. С компьютеров в России SSH до этого сервера часто обрезается фильтрами
операторов, а из сети Google проходит.

1. Меню «Среда выполнения» → «Выполнить все» (Ctrl+F9). На предупреждение «Этот блокнот создан не
   Google» ответьте «Всё равно запустить».
2. Сначала блокнот без пароля проверит канал до сервера несколькими способами (до двух минут). Когда
   появится поле «Пароль root», введите пароль сервера и нажмите Enter. Пароль остаётся только в памяти
   этой сессии Colab и никуда не записывается.
3. Ждите 5–15 минут, ход установки виден ниже. Если закрыть вкладку, установка на сервере всё равно
   доработает.
4. В конце блокнот предложит сменить пароль root. Это стоит сделать: старый пароль пересылался в чате.

In [ ]:
# Настройки
HOST = "188.120.246.123"
PORT = 22
DOMAIN = "familyapp1.ru"
INSTALLER = "https://raw.githubusercontent.com/MorfNNNNN/ubt-factory/claude/cool-gates-wyu9vs/shield/deploy/install.sh"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paramiko"], check=True)

import base64, getpass, hashlib, io, logging, re, socket, time, urllib.request
import paramiko

logging.getLogger("paramiko").setLevel(logging.CRITICAL)  # failed attempts are reported below in plain words
CLIENT_ID = "SSH-2.0-OpenSSH_9.6"
# Some routes to the server pass the SSH greeting and then stall the session. Smaller packets and a
# greeting sent in pieces get through such filters more often, so several ways are tried in turn.
STRATEGIES = [
    ("обычное соединение", None, False),
    ("пакеты до 1000 байт", 1000, False),
    ("пакеты до 536 байт, приветствие по частям", 536, True),
    ("пакеты до 88 байт, приветствие по частям", 88, True),
]


class Trickle:
    """Socket wrapper that sends the first bytes of the session two at a time."""

    def __init__(self, sock, first=64):
        self._sock, self._left = sock, first

    def send(self, data):
        if self._left <= 0:
            return self._sock.send(data)
        sent = self._sock.send(data[:min(2, self._left)])
        self._left -= sent
        time.sleep(0.05)
        return sent

    def sendall(self, data):
        while data:
            data = data[self.send(data):]

    def __getattr__(self, name):
        return getattr(self._sock, name)


def open_socket(mss, trickle):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        if mss:
            sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_MAXSEG, mss)  # announced in SYN: the server sends smaller packets too
        sock.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        sock.settimeout(30)
        sock.connect((HOST, PORT))
    except OSError:
        sock.close()
        raise
    return Trickle(sock) if trickle else sock


def handshake(mss, trickle, timeout):
    """Opens an SSH session up to the password step. Returns (transport or None, what happened)."""
    try:
        sock = open_socket(mss, trickle)
    except OSError as e:
        return None, f"нет TCP-соединения ({e})"
    t = paramiko.Transport(sock)
    t.local_version = CLIENT_ID
    t.banner_timeout = t.handshake_timeout = timeout
    try:
        # start_client returns quietly when its own timeout runs out, so give the transport's timers
        # room to fail first and check that the key exchange really finished.
        t.start_client(timeout=timeout + 15)
        if not t.is_active() or t.session_id is None:
            raise EOFError()
        return t, f"работает, сервер {t.remote_version}"
    except Exception as e:  # EOFError or SSHException when the session stalls
        t.close()
        if not t.remote_version:
            return None, "сервер не прислал приветствие SSH"
        return None, f"приветствие {t.remote_version} пришло, затем сеанс оборвался ({type(e).__name__})"


def connect(password):
    """Logs in as root by the ways that passed the check; retries while the route is flaky."""
    for name, mss, trickle in working * 3:
        t, result = handshake(mss, trickle, 120)
        if t is None:
            print(f"{name}: {result}, пробую ещё раз")
            time.sleep(10)
            continue
        try:
            t.auth_password("root", password)
            return t
        except paramiko.BadAuthenticationType:
            t.close()
            raise SystemExit("На сервере выключен вход по паролю. Сообщите об этом Claude.")
        except paramiko.AuthenticationException:
            t.close()
            raise SystemExit("Пароль не подошёл. Запустите эту ячейку ещё раз и введите пароль root заново.")
        except Exception as e:
            t.close()
            print(f"{name}: сеанс оборвался при входе ({type(e).__name__}), пробую ещё раз")
            time.sleep(10)
    raise SystemExit("Не удалось войти на сервер. Скопируйте строки выше и отправьте Claude.")


def run(t, command, stdin_text=None):
    """Runs a command on the server; returns (exit code, stdout bytes, stderr bytes)."""
    ch = t.open_session(timeout=60)
    ch.settimeout(120)
    ch.exec_command(command)
    if stdin_text is not None:
        ch.sendall(stdin_text.encode())
    ch.shutdown_write()
    stdout, stderr = ch.makefile("rb").read(), ch.makefile_stderr("rb").read()
    code = ch.recv_exit_status()
    ch.close()
    return code, stdout, stderr


print("Проверяю канал до сервера, пароль пока не нужен:")
working = []
for name, mss, trickle in STRATEGIES:
    t, result = handshake(mss, trickle, 30)
    print(f"  {name}: {result}")
    if t:
        t.close()
        working.append((name, mss, trickle))
if not working:
    raise SystemExit("Из сети Google SSH до сервера обрывается при любом способе. Скопируйте строки выше и отправьте Claude.")

password = getpass.getpass("Пароль root: ")
ssh = connect(password)
key = ssh.get_remote_server_key()
fingerprint = base64.b64encode(hashlib.sha256(key.asbytes()).digest()).decode().rstrip("=")
print(f"Подключено к {HOST}. Отпечаток ключа сервера: {key.get_name()} SHA256:{fingerprint}")

In [ ]:
# Запуск установщика и его журнал
ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")

_, state, _ = run(ssh, 'if [ -f /root/install.pid ] && kill -0 "$(cat /root/install.pid)" 2>/dev/null; then echo running; fi')
if b"running" in state:
    print("Установка уже идёт, показываю её журнал.\n")
else:
    script = urllib.request.urlopen(INSTALLER, timeout=60).read()
    sftp = paramiko.SFTPClient.from_transport(ssh)
    sftp.putfo(io.BytesIO(script), "/root/install.sh")
    sftp.close()
    # nohup and a pid file: the installer keeps running even if this SSH session drops.
    _, state, err = run(ssh, f"(nohup bash -c 'bash /root/install.sh {DOMAIN}; echo INSTALL_EXIT=$?' "
                             "</dev/null >/root/install.log 2>&1 & echo $! >/root/install.pid); echo started")
    if b"started" not in state:
        raise SystemExit("Не удалось запустить установщик: " + err.decode(errors="replace"))
    print("Установка запущена, это 5–15 минут. Вкладку можно закрыть: установка на сервере доработает сама.\n")

offset, pending, exit_code = 0, b"", None
deadline = time.time() + 45 * 60
while exit_code is None and time.time() < deadline:
    time.sleep(10)
    try:
        _, chunk, _ = run(ssh, f"tail -c +{offset + 1} /root/install.log")
    except Exception:
        ssh = connect(password)
        continue
    offset += len(chunk)
    lines = (pending + chunk).split(b"\n")
    pending = lines.pop()
    for line in lines:
        text = ANSI.sub("", line.decode(errors="replace")).split("\r")[-1]
        if text.startswith("INSTALL_EXIT="):
            exit_code = int(text.split("=", 1)[1])
        else:
            print(text, flush=True)

if exit_code is None:
    print("\nУстановка идёт дольше 45 минут. Запустите эту ячейку ещё раз, чтобы снова смотреть журнал.")
elif exit_code == 0:
    print("\nГотово: сервер установлен.")
else:
    raise SystemExit(f"Установщик остановился с ошибкой (код {exit_code}). Скопируйте последние строки журнала выше и отправьте Claude.")

In [ ]:
# Проверка сайта
for path in ["/health", "/privacy", "/terms"]:
    url = f"https://{DOMAIN}{path}"
    try:
        with urllib.request.urlopen(url, timeout=20) as response:
            print(f"{url} → {response.status}")
    except Exception as e:
        print(f"{url} → не открылся ({e}). Если сертификат ещё выпускается, проверьте через пару минут.")

In [ ]:
# Смена пароля root
new_password = getpass.getpass("Новый пароль root, от 12 символов (Enter — пропустить): ")
if not new_password:
    print("Пароль не изменён. Смените его позже: команда passwd на сервере.")
elif len(new_password) < 12 or "\n" in new_password:
    print("Нужен пароль от 12 символов. Пароль не изменён, запустите ячейку ещё раз.")
elif new_password != getpass.getpass("Повторите новый пароль: "):
    print("Пароли не совпали. Пароль не изменён, запустите ячейку ещё раз.")
else:
    code, _, err = run(ssh, "chpasswd", stdin_text=f"root:{new_password}\n")
    if code == 0:
        password = new_password
        print("Пароль root изменён. Сохраните его в надёжном месте и никому не пересылайте.")
    else:
        print("Не получилось сменить пароль: " + err.decode(errors="replace"))
ssh.close()

Готово. Напишите Claude «готово»: он проверит сайт из российских сетей и продолжит публикацию в RuStore.
Дальше SSH не нужен: сервер сам забирает обновления с GitHub каждые 10 минут.